# Лабораторная работа № 1

**Тема:** Julia. Установка и настройка. Основные принципы.

**Цель:** подготовить рабочее пространство и инструментарий для Julia,
познакомиться с основами синтаксиса языка и выполнить примеры работы с
вводом-выводом, преобразованием типов, векторами и матрицами.

In [1]:
using DrWatson
@quickactivate "project"

GR строит графики без открытия окна, что позволяет одинаково запускать
лабораторную из терминала, Jupyter и при автоматической генерации документа.

In [2]:
ENV["GKSwstype"] = "100"

using BenchmarkTools
using CSV
using DataFrames
using Dates
using DelimitedFiles
using DifferentialEquations
using JLD2
using LinearAlgebra
using Plots

include(srcdir("lab01.jl"))

Main.var"##277".calculate_for_parameter

## Проверка рабочего окружения

In [3]:
println("Версия Julia: ", VERSION)
println("Дата выполнения: ", today())
println("Активный проект: ", Base.active_project())
println("Корень проекта: ", projectdir())

Версия Julia: 1.12.5
Дата выполнения: 2026-09-08
Активный проект: /home/asvedjmina/university/static_analys/labs/lab01/project/Project.toml
Корень проекта: /home/asvedjmina/university/static_analys/labs/lab01/project


## Типы числовых величин

Функция `typeof` возвращает конкретный тип значения.

In [4]:
values = (3, 3.5, 3 / 3.5, 3 + 4im, pi)

for value in values
    println(repr(value), " имеет тип ", typeof(value))
end

3 имеет тип Int64
3.5 имеет тип Float64
0.8571428571428571 имеет тип Float64
3 + 4im имеет тип Complex{Int64}
π имеет тип Irrational{:π}


Вещественные числа поддерживают специальные значения бесконечности и
неопределённости.

In [5]:
special_values = (Inf, -Inf, NaN, 1.0 / 0.0, 0.0 / 0.0)

for value in special_values
    println(repr(value), " имеет тип ", typeof(value))
end

Inf имеет тип Float64
-Inf имеет тип Float64
NaN имеет тип Float64
Inf имеет тип Float64
NaN имеет тип Float64


## Границы целочисленных типов

In [6]:
integer_types = [
    Int8,
    Int16,
    Int32,
    Int64,
    Int128,
    UInt8,
    UInt16,
    UInt32,
    UInt64,
    UInt128,
]

for T in integer_types
    println(lpad(string(T), 7), ": [", typemin(T), ", ", typemax(T), "]")
end

   Int8: [-128, 127]
  Int16: [-32768, 32767]
  Int32: [-2147483648, 2147483647]
  Int64: [-9223372036854775808, 9223372036854775807]
 Int128: [-170141183460469231731687303715884105728, 170141183460469231731687303715884105727]
  UInt8: [0, 255]
 UInt16: [0, 65535]
 UInt32: [0, 4294967295]
 UInt64: [0, 18446744073709551615]
UInt128: [0, 340282366920938463463374607431768211455]


## Преобразование и продвижение типов

In [7]:
direct_conversion = (Int64(2.0), Char(65))
generic_conversion = (convert(Int64, 2.0), convert(Char, 65))
boolean_conversion = (Bool(1), Bool(0))
promoted = promote(Int8(1), Float16(4.5), Float32(4.1))

println("Прямое преобразование: ", direct_conversion)
println("Преобразование convert: ", generic_conversion)
println("Преобразование Bool: ", boolean_conversion)
println("Результат promote: ", promoted)
println("Типы после promote: ", typeof.(promoted))

Прямое преобразование: (2, 'A')
Преобразование convert: (2, 'A')
Преобразование Bool: (true, false)
Результат promote: (1.0f0, 4.5f0, 4.1f0)
Типы после promote: (Float32, Float32, Float32)


## Определение функций

In [8]:
function square(x)
    return x^2
end

cube(x) = x^3

println("square(4) = ", square(4))
println("cube(3) = ", cube(3))

square(4) = 16
cube(3) = 27


## Одномерные и двумерные массивы

In [9]:
row_vector = [4 7 6]
column_vector = [1, 2, 3]

println("Вектор-строка: ", row_vector)
println("Вектор-столбец: ", column_vector)
println("Вторые элементы: ", row_vector[2], ", ", column_vector[2])

a, b, c, d = 1, 2, 3, 4
Am = [a b; c d]

println("Матрица Am:")
display(Am)
println("Элементы Am: ", (Am[1, 1], Am[1, 2], Am[2, 1], Am[2, 2]))

aa = [1 2]
AA = [1 2; 3 4]
println("aa * AA * aa' = ", aa * AA * aa')

Вектор-строка: [4 7 6]
Вектор-столбец: [1, 2, 3]
Вторые элементы: 7, 2
Матрица Am:
Элементы Am: (1, 2, 3, 4)
aa * AA * aa' = [27;;]


2×2 Matrix{Int64}:
 1  2
 3  4

## Самостоятельная работа: чтение, запись и вывод

`print` не добавляет перевод строки, `println` добавляет, а `show` выводит
представление объекта, близкое к синтаксису Julia.

In [10]:
print("Вывод print; ")
println("вывод println")
show(stdout, "строка через show")
println()

mkpath(datadir("exp_pro"))
text_file = datadir("exp_pro", "example.txt")
binary_file = datadir("exp_pro", "bytes.bin")
table_file = datadir("exp_pro", "table.csv")

open(text_file, "w") do io
    println(io, "первая строка")
    println(io, "вторая строка")
    println(io, "третья строка")
end

whole_text = read(text_file, String)
println("Результат read(..., String):")
println(whole_text)

open(text_file, "r") do io
    println("Результат readline: ", readline(io))
end

all_lines = readlines(text_file)
println("Результат readlines: ", all_lines)

bytes_written = write(binary_file, UInt8[0x41, 0x42, 0x43, 0x0a])
println("Функция write записала байт: ", bytes_written)
println("Байты после read: ", read(binary_file))

numeric_table = [1.0 2.0 3.0; 4.0 5.0 6.0]
writedlm(table_file, numeric_table, ',')
table_from_file = readdlm(table_file, ',', Float64)
println("Таблица после readdlm:")
display(table_from_file)

Вывод print; вывод println
"строка через show"
Результат read(..., String):
первая строка
вторая строка
третья строка

Результат readline: первая строка
Результат readlines: ["первая строка", "вторая строка", "третья строка"]
Функция write записала байт: 4
Байты после read: UInt8[0x41, 0x42, 0x43, 0x0a]
Таблица после readdlm:


2×3 Matrix{Float64}:
 1.0  2.0  3.0
 4.0  5.0  6.0

## Самостоятельная работа: `parse` и `tryparse`

`parse` создаёт значение заданного типа или выбрасывает исключение.
`tryparse` при некорректной строке возвращает `nothing`.

In [11]:
println("parse(Int, \"42\") = ", parse(Int, "42"))
println("parse(Float64, \"3.1415\") = ", parse(Float64, "3.1415"))
println("parse(Bool, \"true\") = ", parse(Bool, "true"))
println("Двоичное 1010 = ", parse(Int, "1010"; base=2))
println("Шестнадцатеричное ff = ", parse(Int, "ff"; base=16))
println("tryparse(Int, \"123\") = ", tryparse(Int, "123"))
println("tryparse(Int, \"abc\") = ", tryparse(Int, "abc"))

parse(Int, "42") = 42
parse(Float64, "3.1415") = 3.1415
parse(Bool, "true") = true
Двоичное 1010 = 10
Шестнадцатеричное ff = 255
tryparse(Int, "123") = 123
tryparse(Int, "abc") = nothing


## Самостоятельная работа: математические операции

In [12]:
x = 7
y = 3

println("x + y = ", x + y)
println("x - y = ", x - y)
println("x * y = ", x * y)
println("x / y = ", x / y)
println("div(x, y) = ", div(x, y))
println("rem(x, y) = ", rem(x, y))
println("x^y = ", x^y)
println("sqrt(9.0) = ", sqrt(9.0))
println("sqrt(-1 + 0im) = ", sqrt(-1 + 0im))

println("Сравнения: ", (x > y, x < y, x == y, x != y, x >= y))

p = true
q = false
println("p && q = ", p && q)
println("p || q = ", p || q)
println("!p = ", !p)

x + y = 10
x - y = 4
x * y = 21
x / y = 2.3333333333333335
div(x, y) = 2
rem(x, y) = 1
x^y = 343
sqrt(9.0) = 3.0
sqrt(-1 + 0im) = 0.0 + 1.0im
Сравнения: (true, false, false, true, true)
p && q = false
p || q = true
!p = false


## Самостоятельная работа: векторы и матрицы

In [13]:
v = [1.0, 2.0, 3.0]
w = [4.0, 5.0, 6.0]
A = [1.0 2.0 3.0; 4.0 5.0 6.0; 7.0 8.0 10.0]
B = [2.0 0.0 1.0; 1.0 3.0 2.0; 0.0 1.0 4.0]

println("v + w = ", v + w)
println("v - w = ", v - w)
println("dot(v, w) = ", dot(v, w))
println("transpose(v) = ", transpose(v))
println("2v = ", 2 .* v)
println("A + B =")
display(A + B)
println("A - B =")
display(A - B)
println("A * v = ", A * v)
println("A * B =")
display(A * B)
println("Поэлементное A .* B =")
display(A .* B)

v + w = [5.0, 7.0, 9.0]
v - w = [-3.0, -3.0, -3.0]
dot(v, w) = 32.0
transpose(v) = transpose([1.0, 2.0, 3.0])
2v = [2.0, 4.0, 6.0]
A + B =
A - B =
A * v = [14.0, 32.0, 53.0]
A * B =
Поэлементное A .* B =


3×3 Matrix{Float64}:
 3.0  2.0   4.0
 5.0  8.0   8.0
 7.0  9.0  14.0

3×3 Matrix{Float64}:
 -1.0  2.0  2.0
  3.0  2.0  4.0
  7.0  7.0  6.0

3×3 Matrix{Float64}:
  4.0   9.0  17.0
 13.0  21.0  38.0
 22.0  34.0  63.0

3×3 Matrix{Float64}:
 2.0   0.0   3.0
 4.0  15.0  12.0
 0.0   8.0  40.0

## Вычисление для набора параметров

Для каждого значения параметра $\alpha$ масштабируем матрицу, умножаем её
на вектор и вычисляем норму матрицы.

In [14]:
parameters = [0.5, 1.0, 2.0]
parameter_results = [
    calculate_for_parameter(alpha, A, v) for alpha in parameters
]

for result in parameter_results
    println(
        "alpha=", result.alpha,
        ", норма=", round(result.matrix_norm; digits=3),
        ", результат=", result.result,
    )
end

alpha=0.5, норма=8.718, результат=[7.0, 16.0, 26.5]
alpha=1.0, норма=17.436, результат=[14.0, 32.0, 53.0]
alpha=2.0, норма=34.871, результат=[28.0, 64.0, 106.0]


Представим результаты в таблице и сохраним в воспроизводимых форматах.

In [15]:
results_table = DataFrame(
    alpha=[result.alpha for result in parameter_results],
    matrix_norm=[result.matrix_norm for result in parameter_results],
    result_1=[result.result[1] for result in parameter_results],
    result_2=[result.result[2] for result in parameter_results],
    result_3=[result.result[3] for result in parameter_results],
)

println("Таблица результатов для набора параметров:")
display(results_table)

csv_file = datadir("exp_pro", "parameter_results.csv")
jld2_file = datadir("sims", "parameter_results.jld2")
CSV.write(csv_file, results_table)
@save jld2_file parameters parameter_results results_table

println("CSV сохранён: ", csv_file)
println("JLD2 сохранён: ", jld2_file)

Таблица результатов для набора параметров:
CSV сохранён: /home/asvedjmina/university/static_analys/labs/lab01/project/data/exp_pro/parameter_results.csv
JLD2 сохранён: /home/asvedjmina/university/static_analys/labs/lab01/project/data/sims/parameter_results.jld2


Row,alpha,matrix_norm,result_1,result_2,result_3
,Float64,Float64,Float64,Float64,Float64
1,0.5,8.7178,7.0,16.0,26.5
2,1.0,17.4356,14.0,32.0,53.0
3,2.0,34.8712,28.0,64.0,106.0


Построим график зависимости нормы масштабированной матрицы от параметра.

In [16]:
mkpath(plotsdir())
parameter_plot = plot(
    results_table.alpha,
    results_table.matrix_norm;
    marker=:circle,
    linewidth=2,
    xlabel="alpha",
    ylabel="Норма матрицы",
    label="norm(alpha * A)",
    title="Вычисление для набора параметров",
)
plot_file = plotsdir("parameter_norms.png")
savefig(parameter_plot, plot_file)
println("График сохранён: ", plot_file)

График сохранён: /home/asvedjmina/university/static_analys/labs/lab01/project/plots/parameter_norms.png


## Пример решения обыкновенного дифференциального уравнения

Решим задачу $u'=-u$, $u(0)=1$ на отрезке $[0,5]$.

In [17]:
decay!(du, u, parameter, time) = (du[1] = -parameter * u[1])
ode_problem = ODEProblem(decay!, [1.0], (0.0, 5.0), 1.0)
ode_solution = solve(ode_problem, Tsit5(); saveat=1.0)
println("Решение ОДУ в узлах: ", first.(ode_solution.u))

Решение ОДУ в узлах: [1.0, 0.3678796381978341, 0.13533581256777455, 0.04978770787351412, 0.018315275151814518, 0.006738503575123539]


## Простейшее измерение времени

In [18]:
elapsed = @belapsed sum($v)
println("Оценка времени sum(v), секунд: ", elapsed)

Оценка времени sum(v), секунд: 2.5619999999999996e-9


## Вывод

Рабочая среда Julia и структура DrWatson подготовлены. На примерах изучены
типы данных, функции, массивы, ввод-вывод, преобразование строк, базовая
математика и операции линейной алгебры. Один литературный исходник позволяет
воспроизводимо получить чистый код, Jupyter Notebook и документацию Quarto.